In [ ]:
# Configuration Module - All Paths, Parameters and Style Settings
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
import warnings
import logging
import pickle  

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Path Settings
DATA_PATH = "Data_new.csv"
MODEL_SAVE_DIR = "saved_models"
PLOT_SAVE_DIR = "saved_plots"
BEST_PARAMS_FILE = "best_params_RF.json"
INPUT_CSV_PATH = "new_data.csv"
OUTPUT_CSV_PATH = "predicted_results.csv"
FOLD_RESULTS_DIR = "fold_results"
CROSS_VALIDATION_DIR = "cross_validation_results"
RESULTS_DATA_FILE = "results_data.pkl"  

# Create save directories
for dir_path in [MODEL_SAVE_DIR, PLOT_SAVE_DIR, FOLD_RESULTS_DIR, CROSS_VALIDATION_DIR]:
    if not os.path.exists(dir_path):
        os.makedirs(dir_path)
        logging.info(f"Directory created: {dir_path}")

# RF (Random Forest) Parameter Ranges - For hyperparameter screening
PARAM_BOUNDS = {
    'n_estimators': (100, 1000),       
    'max_depth': (3, 20),              
    'min_samples_split': (2, 20),       
    'min_samples_leaf': (1, 10),       
    'max_features': (0.1, 1.0),        
    'bootstrap': (True, False)        
}

# Default Best Hyperparameters (specified by user)
DEFAULT_BEST_PARAMS = {
    'n_estimators': 975,
    'max_depth': 8,
    'min_samples_split': 2,
    'min_samples_leaf': 1,
    'max_features': 0.9696713012007199,
    'bootstrap': True,
    'random_state': 42,
    'oob_score': True
}

# Plot Style Settings
# Ignore font warnings
warnings.filterwarnings("ignore", category=UserWarning, message="Glyph.*")

# Set fonts that support Chinese (retained for compatibility if Chinese labels are needed)
plt.rcParams['font.family'] = ['Arial', 'SimHei', 'WenQuanYi Micro Hei', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

# Global Style Settings
sns.set_style("whitegrid")
plt.rcParams.update({
    'font.family': ['Arial', 'SimHei'],
    'font.size': 8,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 7,
    'ytick.labelsize': 7,
    'legend.fontsize': 7,
    'figure.dpi': 300,
    'figure.figsize': (3.3, 3),
    'lines.linewidth': 1,
    'axes.linewidth': 0.8,
    'grid.linewidth': 0.5
})

# Nature Journal Color Scheme
NATURE_COLORS = [
    '#403990',  # Dark blue
    '#80A6E2',  # Light blue
    '#FBDD85',  # Yellow
    '#F46F43',  # Orange
    '#CF3D3E'   # Red
]

COLORS = {
    'primary': NATURE_COLORS[0],
    'secondary': NATURE_COLORS[3],
    'tertiary': NATURE_COLORS[2],
    'neutral': '#636363',
    'light': '#F7F7F7',
    'dark': '#252525',
    'ci_95': '#A6CEE3',
    'ci_90': '#B2DF8A',
    'cmap': LinearSegmentedColormap.from_list("nature_cmap", NATURE_COLORS)
}

# Data Loading Function
def load_data():
    """Load and prepare data"""
    try:
        data_df = pd.read_csv(DATA_PATH, encoding='gbk')
        logging.info(f"Dataset loaded successfully, total {len(data_df)} records and {len(data_df.columns)} features")
        
        # Define feature columns and target column
        target_column = 'ET1'
        drop_columns = ['No.', 'Acceptor', target_column]
        feature_columns = [col for col in data_df.columns if col not in drop_columns]
        
        # Print selected feature columns and target column for user confirmation
        print("\n===== Feature and Target Column Information =====")
        print(f"Target column: {target_column}")
        print(f"Number of feature columns: {len(feature_columns)}")
        print(f"List of feature columns: {feature_columns}")
        print("===============================================\n")
        
        # Extract features and target variable
        X = data_df[feature_columns]
        y = data_df[target_column]
        
        return X, y, data_df, feature_columns
    except FileNotFoundError:
        logging.error("Error: Dataset file not found, please check the file path.")
        raise

# Function to Save Result Data
def save_results_data(results_data, file_path=RESULTS_DATA_FILE):
    """Save result data for use in other modules"""
    try:
        with open(file_path, 'wb') as f:
            # Exclude non-serializable objects
            serializable_data = {k: v for k, v in results_data.items() 
                                if k not in ['final_model', 'scaler']}
            pickle.dump(serializable_data, f)
        logging.info(f"Result data saved to: {file_path}")
    except Exception as e:
        logging.error(f"Failed to save result data: {e}")

# Function to Load Result Data
def load_results_data(file_path=RESULTS_DATA_FILE):
    """Load result data required by other modules"""
    try:
        with open(file_path, 'rb') as f:
            return pickle.load(f)
    except FileNotFoundError:
        logging.error(f"Result data file not found: {file_path}, please run Module 2 first")
        raise
    except Exception as e:
        logging.error(f"Failed to load result data: {e}")
        raise

# Load data in advance to display feature and target column information
try:
    X, y, data_df, feature_names = load_data()
except Exception as e:
    logging.warning(f"Data preview failed: {e}, will retry in subsequent modules")

In [ ]:
# Module 1: Model Hyperparameter Screening
# Optimizing Random Forest Hyperparameters using Genetic Algorithm

import random
import numpy as np
import pandas as pd
from deap import base, creator, tools, algorithms
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor  

# Load data
X, y, data_df, feature_names = load_data()

# Genetic Algorithm Initialization
def init_ga():
    """Initialize genetic algorithm components"""
    if not hasattr(creator, 'FitnessMax'):
        creator.create("FitnessMax", base.Fitness, weights=(1.0,))
    
    if not hasattr(creator, 'Individual'):
        creator.create("Individual", list, fitness=creator.FitnessMax)
    
    toolbox = base.Toolbox()
    
    # Register parameter generators - Random Forest parameters
    toolbox.register("attr_n_estimators", random.randint, *PARAM_BOUNDS['n_estimators'])
    toolbox.register("attr_max_depth", random.randint, *PARAM_BOUNDS['max_depth'])
    toolbox.register("attr_min_samples_split", random.randint, *PARAM_BOUNDS['min_samples_split'])
    toolbox.register("attr_min_samples_leaf", random.randint, *PARAM_BOUNDS['min_samples_leaf'])
    toolbox.register("attr_max_features", random.uniform, *PARAM_BOUNDS['max_features'])
    toolbox.register("attr_bootstrap", random.choice, PARAM_BOUNDS['bootstrap'])
    
    # Register individual and population generators - Random Forest parameter order
    toolbox.register("individual", tools.initCycle, creator.Individual,
                    (toolbox.attr_n_estimators, toolbox.attr_max_depth, 
                     toolbox.attr_min_samples_split, toolbox.attr_min_samples_leaf,
                     toolbox.attr_max_features, toolbox.attr_bootstrap), n=1)
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    
    return toolbox

def custom_mutate(individual):
    """Custom mutation function - Random Forest parameters"""
    mutation_probability = 0.2
    if random.random() < mutation_probability:
        individual[0] = random.randint(*PARAM_BOUNDS['n_estimators'])  # n_estimators parameter mutation
    if random.random() < mutation_probability:
        individual[1] = random.randint(*PARAM_BOUNDS['max_depth'])  # max_depth parameter mutation
    if random.random() < mutation_probability:
        individual[2] = random.randint(*PARAM_BOUNDS['min_samples_split'])  # min_samples_split parameter mutation
    if random.random() < mutation_probability:
        individual[3] = random.randint(*PARAM_BOUNDS['min_samples_leaf'])  # min_samples_leaf parameter mutation
    if random.random() < mutation_probability:
        individual[4] = random.uniform(*PARAM_BOUNDS['max_features'])  # max_features parameter mutation
    if random.random() < mutation_probability:
        individual[5] = random.choice(PARAM_BOUNDS['bootstrap'])  # bootstrap parameter mutation
    return individual,

def evaluate(params, X, y):
    """Evaluation function using 10-fold cross-validation - Random Forest parameters"""
    param_dict = {
        'n_estimators': params[0],
        'max_depth': params[1],
        'min_samples_split': params[2],
        'min_samples_leaf': params[3],
        'max_features': params[4],
        'bootstrap': params[5],
        'random_state': 42,
        'oob_score': True  # Use out-of-bag score
    }
    
    kf = KFold(n_splits=10, shuffle=True, random_state=42)
    fold_scores = []

    for train_idx, val_idx in kf.split(X):
        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]
        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Random Forest is not sensitive to feature scaling, so we can skip standardization
        model = RandomForestRegressor(**param_dict)  # Use Random Forest model
        model.fit(X_train_fold.values, y_train_fold)
        fold_scores.append(r2_score(y_val_fold, model.predict(X_val_fold.values)))

    return (np.mean(fold_scores),)

def optimize_hyperparameters(X, y, pop_size=50, generations=30, cxpb=0.7, mutpb=0.2):
    """Optimize hyperparameters using genetic algorithm"""
    toolbox = init_ga()
    
    # Register genetic algorithm operations
    toolbox.register("evaluate", evaluate, X=X, y=y)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", custom_mutate)
    toolbox.register("select", tools.selTournament, tournsize=3)
    
    # Initialize population
    population = toolbox.population(n=pop_size)
    
    # Record best individuals during evolution
    best_individuals = []
    
    logging.info("Starting hyperparameter optimization...")
    for gen in range(generations):
        # Select next generation individuals
        offspring = algorithms.varAnd(population, toolbox, cxpb=cxpb, mutpb=mutpb)
        
        # Evaluate new individuals
        fits = toolbox.map(toolbox.evaluate, offspring)
        for fit, ind in zip(fits, offspring):
            ind.fitness.values = fit
        
        # Select next generation population
        population = toolbox.select(offspring, k=len(population))
        
        # Record best individual of current generation
        top_ind = tools.selBest(population, 1)[0]
        best_individuals.append((gen, top_ind, top_ind.fitness.values[0]))
        
        if gen % 5 == 0:
            logging.info(f"Generation {gen} - Best R²: {top_ind.fitness.values[0]:.4f}")
    
    # Find the best individual across all generations
    best_gen, best_ind, best_score = max(best_individuals, key=lambda x: x[2])
    logging.info(f"\nOptimization completed! Best R² score: {best_score:.4f} (in generation {best_gen})")
    
    # Convert best parameters to dictionary - Random Forest parameters
    best_params = {
        'n_estimators': best_ind[0],
        'max_depth': best_ind[1],
        'min_samples_split': best_ind[2],
        'min_samples_leaf': best_ind[3],
        'max_features': best_ind[4],
        'bootstrap': best_ind[5],
        'random_state': 42,
        'oob_score': True  # Use out-of-bag score
    }
    
    return best_params, best_score

# Execute hyperparameter optimization
best_params, best_score = optimize_hyperparameters(X, y)

# Save best parameters
with open(BEST_PARAMS_FILE, 'w') as f:
    json.dump(best_params, f, indent=4)

logging.info(f"Best hyperparameters saved to {BEST_PARAMS_FILE}:")
for param, value in best_params.items():
    logging.info(f"  {param}: {value}")


In [ ]:
# Module 2: Model Training and Testing with Optimal Hyperparameters

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor  
from sklearn.preprocessing import StandardScaler
from joblib import dump
import json
import os
import scipy.stats as stats

# Load data
X, y, data_df, feature_names = load_data()

# Load best parameters; use user-specified default parameters if not found
try:
    with open(BEST_PARAMS_FILE, 'r') as f:
        best_params = json.load(f)
    logging.info(f"Loaded best parameters from {BEST_PARAMS_FILE}")
except FileNotFoundError:
    logging.warning(f"{BEST_PARAMS_FILE} not found, using user-specified default parameters")
    best_params = DEFAULT_BEST_PARAMS

# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=9)
logging.info(f"Training set size: {len(X_train)}, Test set size: {len(X_test)}")

# Random Forest is not sensitive to feature scaling, so standardization can be skipped

# Train the final model - using Random Forest
final_model = RandomForestRegressor(**best_params)
final_model.fit(X_train.values, y_train)

# Save the model (Random Forest does not require a standardizer)
model_path = os.path.join(MODEL_SAVE_DIR, "random_forest_model.joblib")
dump(final_model, model_path)
logging.info(f"Model saved to: {model_path}")

# Calculate predictions and performance metrics
# Test set
y_pred_test = final_model.predict(X_test.values)
test_metrics = {
    'MAE': mean_absolute_error(y_test, y_pred_test),
    'RMSE': np.sqrt(mean_squared_error(y_test, y_pred_test)),
    'R²': r2_score(y_test, y_pred_test),
    'r': np.corrcoef(y_test, y_pred_test)[0][1]
}

# Training set
y_pred_train = final_model.predict(X_train.values)
train_metrics = {
    'MAE': mean_absolute_error(y_train, y_pred_train),
    'RMSE': np.sqrt(mean_squared_error(y_train, y_pred_train)),
    'R²': r2_score(y_train, y_pred_train),
    'r': np.corrcoef(y_train, y_pred_train)[0][1]
}

# Output performance metrics, rounded to 4 decimal places
logging.info(f"\nTest set performance metrics: R²={test_metrics['R²']:.4f}, MAE={test_metrics['MAE']:.4f}, RMSE={test_metrics['RMSE']:.4f}, r={test_metrics['r']:.4f}")
logging.info(f"Training set performance metrics: R²={train_metrics['R²']:.4f}, MAE={train_metrics['MAE']:.4f}, RMSE={train_metrics['RMSE']:.4f}, r={train_metrics['r']:.4f}")

# Get feature importance (Random Forest-specific feature importance representation)
feature_importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': final_model.feature_importances_
})
feature_importance = feature_importance.sort_values('Importance', ascending=False)

# Save feature importance
importance_path = os.path.join(MODEL_SAVE_DIR, "feature_importance.csv")
feature_importance.to_csv(importance_path, index=False)
logging.info(f"Feature importance saved to: {importance_path}")
print("\n===== Feature Importance (Sorted by Importance) =====")
print(feature_importance)

# Function to calculate confidence intervals
def calculate_confidence_interval(model, X, y_true, y_pred, alpha=0.05):
    residuals = y_true - y_pred
    residual_std = np.std(residuals)
    n = len(y_true)
    t_value = stats.t.ppf(1 - alpha/2, n - 2)
    prediction_interval = t_value * residual_std * np.sqrt(1 + 1/n)
    lower_bound = y_pred - prediction_interval
    upper_bound = y_pred + prediction_interval
    return lower_bound, upper_bound, prediction_interval

# Calculate confidence intervals
lower_bound_test, upper_bound_test, _ = calculate_confidence_interval(
    final_model, X_test.values, y_test.values, y_pred_test
)
lower_bound_train, upper_bound_train, _ = calculate_confidence_interval(
    final_model, X_train.values, y_train.values, y_pred_train
)

# Perform 10-fold cross-validation
kf = KFold(n_splits=10, shuffle=True, random_state=42)
fold_metrics = []  # Store test set and training set metrics for each fold
all_y_true = []
all_y_pred = []
all_lower_bounds = []
all_upper_bounds = []
all_indices = []

logging.info("\nStarting 10-fold cross-validation...")
for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
    logging.info(f"\n{'='*20} Fold {fold} {'='*20}")
    X_train_fold = X.iloc[train_idx]
    X_val_fold = X.iloc[val_idx]
    y_train_fold = y.iloc[train_idx]
    y_val_fold = y.iloc[val_idx]

    # Random Forest does not require standardization
    model = RandomForestRegressor(**best_params)
    model.fit(X_train_fold.values, y_train_fold)

    # Make predictions
    y_pred_fold = model.predict(X_val_fold.values)
    y_pred_train_fold = model.predict(X_train_fold.values)  # Calculate training set predictions
    
    # Calculate confidence intervals
    lower_bound_fold, upper_bound_fold, _ = calculate_confidence_interval(
        model, X_val_fold.values, y_val_fold.values, y_pred_fold
    )

    # Calculate test set metrics
    val_mae = mean_absolute_error(y_val_fold, y_pred_fold)
    val_rmse = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
    val_r2 = r2_score(y_val_fold, y_pred_fold)
    val_r = np.corrcoef(y_val_fold, y_pred_fold)[0][1]
    
    # Calculate training set metrics
    train_mae = mean_absolute_error(y_train_fold, y_pred_train_fold)
    train_rmse = np.sqrt(mean_squared_error(y_train_fold, y_pred_train_fold))
    train_r2 = r2_score(y_train_fold, y_pred_train_fold)
    train_r = np.corrcoef(y_train_fold, y_pred_train_fold)[0][1]

    # Record metrics, including both training and test sets
    fold_metrics.append({
        'Fold': fold,
        # Test set metrics
        'Val_MAE': val_mae,
        'Val_RMSE': val_rmse,
        'Val_R²': val_r2,
        'Val_Pearson_r': val_r,
        # Training set metrics
        'Train_MAE': train_mae,
        'Train_RMSE': train_rmse,
        'Train_R²': train_r2,
        'Train_Pearson_r': train_r,
        # Other metrics
        'CI_width_95': np.mean(upper_bound_fold - lower_bound_fold),
    })

    # Print current fold metrics, rounded to 4 decimal places
    logging.info(f"Fold {fold} - Test set: R²={val_r2:.4f}, MAE={val_mae:.4f}, RMSE={val_rmse:.4f}, r={val_r:.4f}")
    logging.info(f"Fold {fold} - Training set: R²={train_r2:.4f}, MAE={train_mae:.4f}, RMSE={train_rmse:.4f}, r={train_r:.4f}")

    # Collect all results
    all_y_true.extend(y_val_fold)
    all_y_pred.extend(y_pred_fold)
    all_lower_bounds.extend(lower_bound_fold)
    all_upper_bounds.extend(upper_bound_fold)
    all_indices.extend(val_idx)

    # Save current fold results
    fold_results_df = pd.DataFrame({
        'No.': data_df.loc[val_idx, 'No.'].values,
        'Acceptor': data_df.loc[val_idx, 'Acceptor'].values,
        'True Value': y_val_fold,
        'Predicted Value': y_pred_fold,
        '95% CI Lower Bound': lower_bound_fold,
        '95% CI Upper Bound': upper_bound_fold,
    })
    fold_results_path = os.path.join(FOLD_RESULTS_DIR, f"fold_{fold}_results.csv")
    fold_results_df.to_csv(fold_results_path, index=False)
    logging.info(f"Fold {fold} results saved to: {fold_results_path}")

# Save cross-validation summary results
cv_results_df = pd.DataFrame({
    'No.': data_df.loc[all_indices, 'No.'].values,
    'Acceptor': data_df.loc[all_indices, 'Acceptor'].values,
    'True Value': all_y_true,
    'Predicted Value': all_y_pred,
    '95% CI Lower Bound': all_lower_bounds,
    '95% CI Upper Bound': all_upper_bounds,
})
cv_results_path = os.path.join(CROSS_VALIDATION_DIR, "cross_validation_results.csv")
cv_results_df.to_csv(cv_results_path, index=False)
logging.info(f"\nCross-validation summary results saved to: {cv_results_path}")

# Save all results for use in subsequent modules
results_data = {
    'X_train': X_train,
    'X_test': X_test,
    'y_train': y_train,
    'y_test': y_test,
    'y_pred_train': y_pred_train,
    'y_pred_test': y_pred_test,
    'lower_bound_train': lower_bound_train,
    'upper_bound_train': upper_bound_train,
    'lower_bound_test': lower_bound_test,
    'upper_bound_test': upper_bound_test,
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'fold_metrics': fold_metrics,
    'all_y_true': all_y_true,
    'all_y_pred': all_y_pred,
    'all_lower_bounds': all_lower_bounds,
    'all_upper_bounds': all_upper_bounds,
    'final_model': final_model,
    'best_params': best_params,
    'feature_names': feature_names,
    'data_df': data_df,
    'feature_importance': feature_importance  # Save feature importance
}

# Save result data to file
save_results_data(results_data)


In [ ]:
# Module 3: Data Export (including various model metrics and saving)

import os
import json
import pandas as pd
import numpy as np

# Custom JSON encoder to handle numpy types
class FloatEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.float64):
            return float(obj)
        return json.JSONEncoder.default(self, obj)

# Get result data from Module 2
try:
    results_data = load_results_data()
    
    # Extract result data from Module 2
    X_train = results_data['X_train']
    X_test = results_data['X_test']
    y_train = results_data['y_train']
    y_test = results_data['y_test']
    y_pred_train = results_data['y_pred_train']
    y_pred_test = results_data['y_pred_test']
    lower_bound_train = results_data['lower_bound_train']
    upper_bound_train = results_data['upper_bound_train']
    lower_bound_test = results_data['lower_bound_test']
    upper_bound_test = results_data['upper_bound_test']
    train_metrics = results_data['train_metrics']
    test_metrics = results_data['test_metrics']
    fold_metrics = results_data['fold_metrics']
    data_df = results_data.get('data_df', None)
    feature_importance = results_data.get('feature_importance', None)  # Get feature importance
    
    logging.info("Successfully loaded model result data")
except NameError:
    logging.error("Model result data not found, please run Module 2 first")
    raise

# Save test set results
if data_df is not None:
    test_results_df = pd.DataFrame({
        'No.': data_df.loc[y_test.index, 'No.'].values,
        'Acceptor': data_df.loc[y_test.index, 'Acceptor'].values,
        'True Value': y_test,
        'Predicted Value': y_pred_test,
        '95% CI Lower Bound': lower_bound_test,
        '95% CI Upper Bound': upper_bound_test
    })
else:
    test_results_df = pd.DataFrame({
        'True Value': y_test,
        'Predicted Value': y_pred_test,
        '95% CI Lower Bound': lower_bound_test,
        '95% CI Upper Bound': upper_bound_test
    })

test_results_path = os.path.join(FOLD_RESULTS_DIR, "test_results.csv")
test_results_df.to_csv(test_results_path, index=False)
logging.info(f"Test set results saved to: {test_results_path}")

# Save training set results
if data_df is not None:
    train_results_df = pd.DataFrame({
        'No.': data_df.loc[y_train.index, 'No.'].values,
        'Acceptor': data_df.loc[y_train.index, 'Acceptor'].values,
        'True Value': y_train,
        'Predicted Value': y_pred_train,
        '95% CI Lower Bound': lower_bound_train,
        '95% CI Upper Bound': upper_bound_train
    })
else:
    train_results_df = pd.DataFrame({
        'True Value': y_train,
        'Predicted Value': y_pred_train,
        '95% CI Lower Bound': lower_bound_train,
        '95% CI Upper Bound': upper_bound_train
    })

train_results_path = os.path.join(FOLD_RESULTS_DIR, "train_results.csv")
train_results_df.to_csv(train_results_path, index=False)
logging.info(f"Training set results saved to: {train_results_path}")

# Save detailed metrics for 10-fold cross-validation (including training set metrics)
cv_metrics_df = pd.DataFrame(fold_metrics)
cv_metrics_path = os.path.join(CROSS_VALIDATION_DIR, "cv_fold_metrics.csv")
cv_metrics_df.to_csv(cv_metrics_path, index=False)
logging.info(f"10-fold cross-validation detailed metrics saved to: {cv_metrics_path}")

# Calculate summary metrics for cross-validation
cv_summary = {
    # Test set summary metrics
    'Mean_Val_MAE': cv_metrics_df['Val_MAE'].mean(),
    'Std_Val_MAE': cv_metrics_df['Val_MAE'].std(),
    'Mean_Val_RMSE': cv_metrics_df['Val_RMSE'].mean(),
    'Std_Val_RMSE': cv_metrics_df['Val_RMSE'].std(),
    'Mean_Val_R²': cv_metrics_df['Val_R²'].mean(),
    'Std_Val_R²': cv_metrics_df['Val_R²'].std(),
    'Mean_Val_Pearson_r': cv_metrics_df['Val_Pearson_r'].mean(),
    'Std_Val_Pearson_r': cv_metrics_df['Val_Pearson_r'].std(),
    # Training set summary metrics
    'Mean_Train_MAE': cv_metrics_df['Train_MAE'].mean(),
    'Std_Train_MAE': cv_metrics_df['Train_MAE'].std(),
    'Mean_Train_RMSE': cv_metrics_df['Train_RMSE'].mean(),
    'Std_Train_RMSE': cv_metrics_df['Train_RMSE'].std(),
    'Mean_Train_R²': cv_metrics_df['Train_R²'].mean(),
    'Std_Train_R²': cv_metrics_df['Train_R²'].std(),
    'Mean_Train_Pearson_r': cv_metrics_df['Train_Pearson_r'].mean(),
    'Std_Train_Pearson_r': cv_metrics_df['Train_Pearson_r'].std(),
    # Other metrics
    'Mean_CI_width_95': cv_metrics_df['CI_width_95'].mean(),
    'Std_CI_width_95': cv_metrics_df['CI_width_95'].std()
}

# Save cross-validation summary metrics
cv_summary_path = os.path.join(CROSS_VALIDATION_DIR, "cv_summary_metrics.csv")
pd.DataFrame([cv_summary]).to_csv(cv_summary_path, index=False)
logging.info(f"Cross-validation summary metrics saved to: {cv_summary_path}")

# Save all performance metrics
metrics_path = os.path.join(CROSS_VALIDATION_DIR, "performance_metrics.json")
all_metrics = {
    'train_metrics': train_metrics,
    'test_metrics': test_metrics,
    'fold_metrics': fold_metrics,
    'cv_summary_metrics': cv_summary
}

with open(metrics_path, 'w') as f:
    json.dump(all_metrics, f, indent=4, cls=FloatEncoder)
logging.info(f"Performance metrics saved to: {metrics_path}")

# Save feature importance (if exists)
if feature_importance is not None:
    importance_path = os.path.join(MODEL_SAVE_DIR, "feature_importance.csv")
    feature_importance.to_csv(importance_path, index=False)
    logging.info(f"Feature importance saved to: {importance_path}")

# Display 10-fold cross-validation metrics table for easy viewing
print("\n===== 10-Fold Cross-Validation Performance Metrics by Fold =====")
print(cv_metrics_df)

print("\n===== 10-Fold Cross-Validation Summary Metrics =====")
summary_df = pd.DataFrame([cv_summary])
print(summary_df)
    

In [ ]:
# Module 4: Visualization Section

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os
try:
    import shap  # Placed in try-except block to prevent errors from missing SHAP library
except ImportError:
    shap = None
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Get result data from Module 2
try:
    results_data = load_results_data()
    
    # Extract required result data
    y_train = results_data['y_train']
    y_test = results_data['y_test']
    y_pred_train = results_data['y_pred_train']
    y_pred_test = results_data['y_pred_test']
    lower_bound_train = results_data['lower_bound_train']
    upper_bound_train = results_data['upper_bound_train']
    lower_bound_test = results_data['lower_bound_test']
    upper_bound_test = results_data['upper_bound_test']
    fold_metrics = results_data['fold_metrics']
    all_y_true = results_data['all_y_true']
    all_y_pred = results_data['all_y_pred']
    all_lower_bounds = results_data['all_lower_bounds']
    all_upper_bounds = results_data['all_upper_bounds']
    X_test = results_data['X_test']
    feature_names = results_data.get('feature_names', X_test.columns.tolist())
    data_df = results_data.get('data_df', None)
    feature_importance = results_data.get('feature_importance', None)  # Get feature importance
    final_model = results_data.get('final_model', None)  # Get final model for SHAP analysis
    
    logging.info("Successfully loaded data required for visualization")
except NameError:
    logging.error("Data required for visualization not found, please run Module 2 first")
    raise

# Scatter plot drawing function
def plot_scatter(y_true, y_pred, lower_bound, upper_bound,
                title, xlabel, ylabel, save_path, show_ci=True, color_map=COLORS['cmap']):
    """Draw scatter plot, optional to show confidence interval"""
    fig, ax = plt.subplots(figsize=(3.3, 3))
    coef = np.polyfit(y_true, y_pred, 1)
    trendline = np.poly1d(coef)
    
    # Calculate axis range
    all_values = np.concatenate([y_true, y_pred])
    if show_ci:
        all_values = np.concatenate([all_values, lower_bound, upper_bound])
    
    data_min = np.min(all_values)
    data_max = np.max(all_values)
    data_range = data_max - data_min
    margin = data_range * 0.1
    min_val = data_min - margin
    max_val = data_max + margin
    
    # Ensure sufficient margin
    if data_range < 0.1:
        min_val = data_min - 0.05
        max_val = data_max + 0.05
    
    # Draw scatter plot
    scatter = ax.scatter(
        y_true, y_pred,
        s=60 * np.abs(y_true - y_pred) / np.max(np.abs(y_true - y_pred)) + 15,
        c=np.abs(y_true - y_pred),
        cmap=color_map,
        alpha=0.7,
        edgecolors='w',
        linewidths=0.5,
        vmin=0, vmax=np.percentile(np.abs(y_true - y_pred), 95)
    )
    
    # Draw trendline
    ax.plot(y_true, trendline(y_true), color=COLORS['secondary'], lw=1.5, linestyle='-',
            label=f'Trend line = {coef[0]:.4f}x + {coef[1]:.4f}')
    
    # Draw confidence interval (if needed)
    if show_ci:
        residuals = y_true - y_pred
        std_error = np.std(residuals)
        x_range = np.array([min_val, max_val])
        upper_line = trendline(x_range) + 1.96 * std_error
        lower_line = trendline(x_range) - 1.96 * std_error
        ax.plot(x_range, upper_line, color=COLORS['primary'], lw=1, linestyle='--', alpha=0.7, label='95% CI')
        ax.plot(x_range, lower_line, color=COLORS['primary'], lw=1, linestyle='--', alpha=0.7)
    
    # Draw ideal line (y=x)
    ax.plot([min_val, max_val], [min_val, max_val],
            color=COLORS['neutral'], lw=1, linestyle=':', label='Ideal line (y=x)')
    
    # Set axes
    ax.set_xlabel(xlabel, labelpad=3)
    ax.set_ylabel(ylabel, labelpad=3)
    ax.xaxis.set_tick_params(width=0.5)
    ax.yaxis.set_tick_params(width=0.5)
    ax.set_xlim(min_val, max_val)
    ax.set_ylim(min_val, max_val)
    
    # Add performance metrics text
    metrics_plot = {
        'R²': r2_score(y_true, y_pred),
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'r': np.corrcoef(y_true, y_pred)[0][1]
    }
    textstr = '\n'.join([
        f'$R^2$ = {metrics_plot["R²"]:.4f}',
        f'MAE = {metrics_plot["MAE"]:.4f}',
        f'RMSE = {metrics_plot["RMSE"]:.4f}',
        f'r = {metrics_plot["r"]:.4f}'
    ])
    ax.text(0.05, 0.95, textstr, transform=ax.transAxes, ha='left', va='top',
            fontsize=8, bbox=dict(facecolor='white', alpha=0.8, edgecolor='none', boxstyle='round,pad=0.3'))
    
    # Add legend
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='lower right', frameon=True, framealpha=0.9, fontsize=6)
    
    # Add color bar
    cbar = fig.colorbar(scatter, ax=ax, pad=0.03, aspect=30)
    cbar.set_label('Absolute Error', rotation=270, labelpad=10, fontsize=8)
    cbar.ax.tick_params(labelsize=6)
    cbar.outline.set_linewidth(0.5)
    
    ax.grid(True, linestyle='--', alpha=0.3)
    plt.title(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches='tight')
    plt.close()
    logging.info(f"Plot saved to: {save_path}")

# Draw test set scatter plot with confidence band
test_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "test_scatter_with_confidence.png")
plot_scatter(
    y_test, y_pred_test, lower_bound_test, upper_bound_test,
    "Test Set Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    test_scatter_with_ci_path,
    show_ci=True
)

# Draw test set scatter plot without confidence band
test_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "test_scatter_without_confidence.png")
plot_scatter(
    y_test, y_pred_test, None, None,
    "Test Set Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    test_scatter_without_ci_path,
    show_ci=False
)

# Draw training set scatter plot with confidence band
train_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "train_scatter_with_confidence.png")
plot_scatter(
    y_train, y_pred_train, lower_bound_train, upper_bound_train,
    "Training Set Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    train_scatter_with_ci_path,
    show_ci=True
)

# Draw training set scatter plot without confidence band
train_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "train_scatter_without_confidence.png")
plot_scatter(
    y_train, y_pred_train, None, None,
    "Training Set Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    train_scatter_without_ci_path,
    show_ci=False
)

# Draw cross-validation boxplot
cv_boxplot_path = os.path.join(PLOT_SAVE_DIR, "cv_boxplot.png")
plt.rcParams.update({'font.size': 8})
fig, axes = plt.subplots(1, 3, figsize=(9.9, 3))

# MAE and RMSE boxplot
data1 = {'Validation MAE': [m['Val_MAE'] for m in fold_metrics], 
         'Validation RMSE': [m['Val_RMSE'] for m in fold_metrics]}
df1 = pd.DataFrame(data1)
sns.boxplot(data=df1, palette=[COLORS['primary'], COLORS['secondary']], 
            width=0.6, linewidth=0.7, ax=axes[0])
axes[0].set_ylabel('Value', labelpad=2)
axes[0].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[0].xaxis.grid(False)

# R² and Pearson correlation coefficient boxplot
data2 = {'Validation $R^2$': [m['Val_R²'] for m in fold_metrics], 
         'Validation Pearson r': [m['Val_Pearson_r'] for m in fold_metrics]}
df2 = pd.DataFrame(data2)
sns.boxplot(data=df2, palette=[COLORS['primary'], COLORS['secondary']], 
            width=0.6, linewidth=0.7, ax=axes[1])
axes[1].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[1].xaxis.grid(False)

# 95% confidence interval width boxplot
data3 = {'95% CI Width': [m['CI_width_95'] for m in fold_metrics]}
df3 = pd.DataFrame(data3)
sns.boxplot(data=df3, palette=[COLORS['primary']], 
            width=0.6, linewidth=0.7, ax=axes[2])
axes[2].set_ylabel('95% CI Width', labelpad=2)
axes[2].yaxis.grid(True, linestyle='--', alpha=0.6)
axes[2].xaxis.grid(False)

# Add median values on boxplots
for i, col in enumerate(df1.columns):
    median = df1[col].median()
    axes[0].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

for i, col in enumerate(df2.columns):
    median = df2[col].median()
    axes[1].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

for i, col in enumerate(df3.columns):
    median = df3[col].median()
    axes[2].text(i, median, f'{median:.4f}', ha='center', va='bottom', fontsize=7, color='black',
                 bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

# Add title
fig.suptitle('10-Fold Cross-Validation Metrics', fontsize=10)
plt.tight_layout(rect=[0, 0, 1, 0.96])  # Adjust layout to fit title
plt.savefig(cv_boxplot_path, dpi=600, bbox_inches='tight')
plt.close()
logging.info(f"Cross-validation boxplot saved to: {cv_boxplot_path}")

# Draw training set vs test set metrics comparison plot
comparison_plot_path = os.path.join(PLOT_SAVE_DIR, "train_vs_val_comparison.png")
plt.figure(figsize=(8, 4))

# Extract R² and MAE from training and test sets for comparison
metrics_to_plot = ['R²', 'MAE']
x = np.arange(len(metrics_to_plot))  # Label positions
width = 0.35  # Bar width

# Prepare data
val_metrics = [
    np.mean([m['Val_R²'] for m in fold_metrics]),
    np.mean([m['Val_MAE'] for m in fold_metrics])
]

train_metrics = [
    np.mean([m['Train_R²'] for m in fold_metrics]),
    np.mean([m['Train_MAE'] for m in fold_metrics])
]

# Draw bar chart
rects1 = plt.bar(x - width/2, val_metrics, width, label='Validation', color=COLORS['primary'])
rects2 = plt.bar(x + width/2, train_metrics, width, label='Training', color=COLORS['secondary'])

# Add text labels
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        plt.text(rect.get_x() + rect.get_width()/2., height,
                f'{height:.4f}',
                ha='center', va='bottom', fontsize=7)

autolabel(rects1)
autolabel(rects2)

# Add labels and title
plt.xticks(x, metrics_to_plot)
plt.ylabel('Value')
plt.title('Training vs Validation Metrics Comparison')
plt.legend()

plt.tight_layout()
plt.savefig(comparison_plot_path, dpi=600, bbox_inches='tight')
plt.close()
logging.info(f"Training vs Validation metrics comparison plot saved to: {comparison_plot_path}")

# Draw cross-validation summary scatter plot
cv_scatter_with_ci_path = os.path.join(PLOT_SAVE_DIR, "cv_scatter_with_confidence.png")
plot_scatter(
    np.array(all_y_true), np.array(all_y_pred), 
    np.array(all_lower_bounds), np.array(all_upper_bounds),
    "Cross-Validation Predictions with 95% CI",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    cv_scatter_with_ci_path,
    show_ci=True
)

cv_scatter_without_ci_path = os.path.join(PLOT_SAVE_DIR, "cv_scatter_without_confidence.png")
plot_scatter(
    np.array(all_y_true), np.array(all_y_pred),
    None, None,
    "Cross-Validation Predictions",
    "Experimental ET1 (eV)",
    "Predicted ET1 (eV)",
    cv_scatter_without_ci_path,
    show_ci=False
)

# Draw feature importance bar chart (Random Forest-specific visualization)
if feature_importance is not None:
    importance_plot_path = os.path.join(PLOT_SAVE_DIR, "feature_importance.png")
    plt.figure(figsize=(8, 6))
    
    # Sort by importance
    importance_sorted = feature_importance.sort_values('Importance', ascending=True)
    
    # Create horizontal bar chart
    plt.barh(importance_sorted['Feature'], importance_sorted['Importance'], 
             color=COLORS['primary'])
    
    # Add value labels (rounded to 4 decimal places)
    for i, v in enumerate(importance_sorted['Importance']):
        plt.text(v, i, f' {v:.4f}', va='center', fontsize=7)
    
    plt.xlabel('Importance')
    plt.ylabel('Feature')
    plt.title('Random Forest Feature Importance')
    plt.tight_layout()
    plt.savefig(importance_plot_path, dpi=600, bbox_inches='tight')
    plt.close()
    logging.info(f"Feature importance bar chart saved to: {importance_plot_path}")

# Correlation heatmap
def plot_correlation_heatmap(data, feature_names, save_path):
    """Draw feature correlation heatmap"""
    try:
        # Extract features and target variable
        heatmap_df = data[feature_names + ['ET1']]
        corr = heatmap_df.corr()

        fig, ax = plt.subplots(figsize=(7, 6), dpi=600)
        cmap = COLORS['cmap']
        norm = plt.Normalize(vmin=-1, vmax=1)

        # Draw correlation heatmap
        for i in range(len(corr.columns)):
            for j in range(len(corr.columns)):
                val = corr.iloc[i, j]
                if i > j:  # Use bubbles for lower triangle
                    ax.scatter(
                        j, i, s=500 * abs(val),
                        c=[cmap(norm(val))], alpha=0.9,
                        edgecolors='w', linewidths=0.5
                    )
                elif i < j:  # Use text for upper triangle
                    text_color = 'white' if abs(val) > 0.5 else 'black'
                    ax.text(
                        j, i, f'{val:.4f}',
                        ha='center', va='center', color=text_color, fontsize=7,
                        bbox={'facecolor': cmap(norm(val)), 'alpha': 0.8, 'pad': 2}
                    )

        # Set axes
        ax.set_xticks(np.arange(len(corr.columns)))
        ax.set_xticklabels(corr.columns, rotation=45, ha='right', fontsize=7)
        ax.set_yticks(np.arange(len(corr.columns)))
        ax.set_yticklabels(corr.columns, fontsize=7)

        # Add grid lines
        ax.set_xticks(np.arange(len(corr.columns)) - 0.5, minor=True)
        ax.set_yticks(np.arange(len(corr.columns)) - 0.5, minor=True)
        ax.grid(which='minor', color='lightgray', linestyle='-', linewidth=0.5)

        # Add color bar
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, shrink=0.8, aspect=30)
        cbar.ax.tick_params(labelsize=7)
        cbar.set_label('Pearson Correlation', fontsize=8)

        plt.title("Feature Correlation Matrix", fontsize=10, pad=15)
        plt.tight_layout()
        plt.savefig(save_path, dpi=600, bbox_inches='tight')
        plt.close()
        logging.info(f"Correlation heatmap saved to: {save_path}")
    except Exception as e:
        logging.error(f"Failed to draw correlation heatmap: {e}")

# Draw correlation heatmap (if data exists)
if data_df is not None:
    corr_heatmap_path = os.path.join(PLOT_SAVE_DIR, 'corr_heatmap.png')
    plot_correlation_heatmap(data_df, feature_names, corr_heatmap_path)

# If SHAP library is installed, draw SHAP value summary plots (Random Forest-specific visualization)
if shap is not None and final_model is not None:
    try:
        # Create SHAP explainer
        explainer = shap.TreeExplainer(final_model)
        shap_values = explainer.shap_values(X_test)
        
        # Draw SHAP summary plot
        shap_summary_plot_path = os.path.join(PLOT_SAVE_DIR, "shap_summary.png")
        plt.figure(figsize=(8, 6))
        shap.summary_plot(shap_values, X_test, feature_names=feature_names, plot_type="bar")
        plt.tight_layout()
        plt.savefig(shap_summary_plot_path, dpi=600, bbox_inches='tight')
        plt.close()
        logging.info(f"SHAP value summary plot saved to: {shap_summary_plot_path}")
        
        # Draw beeswarm plot
        shap_beeswarm_plot_path = os.path.join(PLOT_SAVE_DIR, "shap_beeswarm.png")
        plt.figure(figsize=(8, 6))
        shap.summary_plot(shap_values, X_test, feature_names=feature_names)
        plt.tight_layout()
        plt.savefig(shap_beeswarm_plot_path, dpi=600, bbox_inches='tight')
        plt.close()
        logging.info(f"SHAP value beeswarm plot saved to: {shap_beeswarm_plot_path}")
    except Exception as e:
        logging.warning(f"Failed to draw SHAP plots: {e}")

logging.info("\nAll visualization plots generated successfully")


In [ ]:
# Module 5: Model Invocation

import os
import json
import pandas as pd
import numpy as np
from joblib import load
import scipy.stats as stats

def load_model(model_dir=MODEL_SAVE_DIR):
    """Load the trained model (Random Forest doesn't require a standardizer)"""
    model_path = os.path.join(model_dir, "random_forest_model.joblib")
    
    try:
        model = load(model_path)
        logging.info(f"Successfully loaded model: {model_path}")
        return model
    except FileNotFoundError as e:
        logging.error(f"Model file not found: {e}")
        raise
    except Exception as e:
        logging.error(f"Error loading model: {e}")
        raise

def predict_new_data(model, new_data, feature_names):
    """Predict new data using the model"""
    try:
        # Ensure the feature order of new data matches the training data
        new_data_ordered = new_data[feature_names]
        
        # Random Forest doesn't require standardization
        
        # Make predictions
        predictions = model.predict(new_data_ordered.values)
        
        # Calculate confidence intervals
        try:
            # Get training data predictions and true values from saved results
            results_data = load_results_data()
            residuals_std = np.std(results_data['y_pred_train'] - results_data['y_train'])
        except Exception as e:
            logging.warning(f"Unable to calculate residual standard deviation, using default value: {e}")
            residuals_std = 0.5  # Provide a reasonable default value
            
        n = len(new_data)
        t_value = stats.t.ppf(0.975, n - 2)  # 95% confidence interval
        prediction_interval = t_value * residuals_std * np.sqrt(1 + 1/n)
        
        lower_bound = predictions - prediction_interval
        upper_bound = predictions + prediction_interval
        
        return predictions, lower_bound, upper_bound
    except KeyError as e:
        logging.error(f"New data is missing required features: {e}")
        raise
    except Exception as e:
        logging.error(f"Error predicting new data: {e}")
        raise

def load_new_data(file_path=INPUT_CSV_PATH):
    """Load new data for prediction"""
    try:
        new_data = pd.read_csv(file_path, encoding='gbk')
        logging.info(f"Successfully loaded new data: {file_path}, {len(new_data)} records total")
        return new_data
    except FileNotFoundError:
        logging.error(f"New data file not found: {file_path}")
        raise
    except Exception as e:
        logging.error(f"Error loading new data: {e}")
        raise

def save_predictions(new_data, predictions, lower_bound, upper_bound, output_path=OUTPUT_CSV_PATH):
    """Save prediction results"""
    try:
        # Create results DataFrame
        result_df = new_data.copy()
        result_df['Predicted ET1'] = predictions
        result_df['95% CI Lower Bound'] = lower_bound
        result_df['95% CI Upper Bound'] = upper_bound
        
        # Save results
        result_df.to_csv(output_path, index=False, encoding='gbk')
        logging.info(f"Prediction results saved to: {output_path}")
        return result_df
    except Exception as e:
        logging.error(f"Error saving prediction results: {e}")
        raise

# Main function: Load model and predict new data
def main():
    # Get feature names from configuration (or from previous modules)
    try:
        # Try to get feature names from Module 2 results
        results_data = load_results_data()
        feature_names = results_data.get('feature_names', None)
    except NameError:
        # If Module 2 hasn't run, get feature names from data
        try:
            _, _, _, feature_names = load_data()
        except Exception as e:
            logging.error(f"Failed to get feature names: {e}")
            # You can also manually specify feature names here
            # feature_names = ['feature1', 'feature2', ...]
            return
    
    if feature_names is None:
        logging.error("Feature names not found, please ensure data is loaded correctly")
        return
    
    # Load model (Random Forest doesn't require a standardizer)
    model = load_model()
    
    # Load new data
    new_data = load_new_data()
    
    # Make predictions
    predictions, lower_bound, upper_bound = predict_new_data(model, new_data, feature_names)
    
    # Save prediction results
    result_df = save_predictions(new_data, predictions, lower_bound, upper_bound)
    
    # Display first 5 prediction results
    logging.info("\nFirst 5 prediction results:")
    print(result_df.head())

if __name__ == "__main__":
    main()
    